Covariates for Background Model for Significantly Recurrent Bins

This notebook prepares data from various genomic features to create a background model for the distribution of structural variants across the genome.

1. Replication timing data: IMR90 (fibroblast CL from fetal lung tissue) (https://hgdownload.cse.ucsc.edu/goldenpath/hg19/encodeDCC/wgEncodeUwRepliSeq/wgEncodeUwRepliSeqImr90WaveSignalRep1.bigWig
) -> lower values mean later replication timing, lifted over to hg38
2. GC content: GC % 5-Base Windows (https://genome.ucsc.edu/cgi-bin/hgTrackUi?hgsid=1758253012_6tWocRwjX4LNth9YX4cHS7IqVGkx&db=hg38&c=chr2&g=gc5BaseBw)
3. Mappability scores: UMap multi-track mappability scores 100mer (https://hgdownload.soe.ucsc.edu/gbdb/hg38/hoffmanMappability/)
4. Chromatin state data: Roadmap Epigenomics Consortium E129 (Osteoblast Primary CL) (https://egg2.wustl.edu/roadmap/data/byFileType/chromhmmSegmentations/ChmmModels/coreMarks/jointModel/final/)
5. RepeatMasker data: UCSC Table Browser (https://genome.ucsc.edu/cgi-bin/hgTables?db=hg38&hgta_group=rep&hgta_track=rmsk&hgta_table=rmsk&hgta_doSchema=describe+table+schema)
6. Fragile sites

Potential future features to add: DNAase (hypersensitivity sites), Epigenomic features, Gene density 

Also, might want to find a way to "fill in" the bins that don't have covariate data

In [1]:
import pandas as pd
import numpy as np
import pyBigWig
from sklearn.preprocessing import StandardScaler
import os

In [2]:
def load_rt(filename):
    genome_bins = pd.read_csv('../data/genome_bins.bed', sep='\t')
    
    rt_data = pd.read_csv(filename, sep='\t', header=None,
                          names=['chromosome', 'start', 'end', 'replication_timing'])
    # strip the 'chr' from the chromosome column
    rt_data['chromosome'] = rt_data['chromosome'].str.replace('chr', '', regex=False)

    # use midpoint
    rt_data['coordinate'] = (rt_data['start'] + rt_data['end']) // 2
    
    results = []
    for chrom in genome_bins['chrom'].unique():
        chrom_bins = genome_bins[genome_bins['chrom'] == chrom].copy()
        chrom_rt = rt_data[rt_data['chromosome'] == chrom].copy()
        
        if len(chrom_rt) == 0:
            print(f"No RT data for chromosome {chrom}")
            continue
        
        rt_means = []
        for _, bin_row in chrom_bins.iterrows():
            bin_rt = chrom_rt[
                (chrom_rt['coordinate'] >= bin_row['start']) &
                (chrom_rt['coordinate'] < bin_row['end'])
            ]['replication_timing'].mean()
            rt_means.append(bin_rt)
        
        chrom_bins['replication_timing'] = rt_means
        results.append(chrom_bins)
    
    genome_bins_with_rt = pd.concat(results)
    
    missing_rt = genome_bins_with_rt['replication_timing'].isna().sum()
    total_bins = len(genome_bins_with_rt)
    print(f"\nMissing RT data summary:")
    print(f"Total bins: {total_bins}")
    print(f"Bins without RT data: {missing_rt} ({missing_rt/total_bins*100:.2f}%)")
    
    return genome_bins_with_rt

rt_data = load_rt('../data/hg38_ref_files/imr90_RT_hg38.bedGraph')
rt_data.to_csv('../data/imr90_rt_binned.bed', sep='\t', index=False)


Missing RT data summary:
Total bins: 56596
Bins without RT data: 575 (1.02%)


In [3]:
# GC content

def load_gc_content(bigwig_file, chunk_size=1000):
    """
    Process GC content from UCSC's gc5BaseBw file
    """
    # load existing bins
    genome_bins = pd.read_csv('../data/genome_bins.bed', sep='\t')

    bw = pyBigWig.open(bigwig_file)
    
    # Initialize array for GC content
    gc_content = []
    
    # Process bins in chunks to manage memory
    total_bins = len(genome_bins)
    for i in range(0, total_bins, chunk_size):
        chunk = genome_bins.iloc[i:i+chunk_size]
        print(f"Processing bins {i} to {i+len(chunk)} of {total_bins} ({(i/total_bins*100):.1f}%)")
        
        chunk_gc = []
        for _, row in chunk.iterrows():
            try:
                # Get mean GC content for this bin
                chrom = f"chr{row['chrom']}" if not str(row['chrom']).startswith('chr') else row['chrom']
                values = bw.stats(chrom, row['start'], row['end'], type="mean")
                gc = values[0] if values[0] is not None else np.nan
                chunk_gc.append(gc)
            except:
                chunk_gc.append(np.nan)
        
        gc_content.extend(chunk_gc)
    
    bw.close()
    
    # Add GC content to DataFrame
    genome_bins_with_gc = genome_bins.copy()
    genome_bins_with_gc['gc_content'] = gc_content
    
    # Report on missing data
    missing_gc = genome_bins_with_gc['gc_content'].isna().sum()
    total_bins = len(genome_bins_with_gc)
    print(f"\nMissing GC data summary:")
    print(f"Total bins: {total_bins}")
    print(f"Bins without GC data: {missing_gc} ({missing_gc/total_bins*100:.2f}%)")
    print(f"\nGC Content Summary:")
    print(genome_bins_with_gc['gc_content'].describe())
    
    return genome_bins_with_gc

gc_bigwig = "../data/hg38_ref_files/gc5Base.bw"
gc_data = load_gc_content(gc_bigwig, chunk_size=1000)
gc_data.to_csv('../data/gc_content.bed', sep='\t', index=False)

Processing bins 0 to 1000 of 56596 (0.0%)
Processing bins 1000 to 2000 of 56596 (1.8%)
Processing bins 2000 to 3000 of 56596 (3.5%)
Processing bins 3000 to 4000 of 56596 (5.3%)
Processing bins 4000 to 5000 of 56596 (7.1%)
Processing bins 5000 to 6000 of 56596 (8.8%)
Processing bins 6000 to 7000 of 56596 (10.6%)
Processing bins 7000 to 8000 of 56596 (12.4%)
Processing bins 8000 to 9000 of 56596 (14.1%)
Processing bins 9000 to 10000 of 56596 (15.9%)
Processing bins 10000 to 11000 of 56596 (17.7%)
Processing bins 11000 to 12000 of 56596 (19.4%)
Processing bins 12000 to 13000 of 56596 (21.2%)
Processing bins 13000 to 14000 of 56596 (23.0%)
Processing bins 14000 to 15000 of 56596 (24.7%)
Processing bins 15000 to 16000 of 56596 (26.5%)
Processing bins 16000 to 17000 of 56596 (28.3%)
Processing bins 17000 to 18000 of 56596 (30.0%)
Processing bins 18000 to 19000 of 56596 (31.8%)
Processing bins 19000 to 20000 of 56596 (33.6%)
Processing bins 20000 to 21000 of 56596 (35.3%)
Processing bins 2100

In [4]:
# Mappability

def load_mappability(map_bigwig, chunk_size=1000):
    """
    Process mappability scores from UCSC's Umap track
    """
    # load existing bins
    genome_bins = pd.read_csv('../data/genome_bins.bed', sep='\t')

    bw = pyBigWig.open(map_bigwig)
    
    # Initialize array for mappability scores
    map_scores = []
    
    # Process bins in chunks to manage memory
    total_bins = len(genome_bins)
    for i in range(0, total_bins, chunk_size):
        chunk = genome_bins.iloc[i:i+chunk_size]
        print(f"Processing bins {i} to {i+len(chunk)} of {total_bins} ({(i/total_bins*100):.1f}%)")
        
        chunk_map = []
        for _, row in chunk.iterrows():
            try:
                # Get mean mappability for this bin
                chrom = f"chr{row['chrom']}" if not str(row['chrom']).startswith('chr') else row['chrom']
                values = bw.stats(chrom, row['start'], row['end'], type="mean")
                map_score = values[0] if values[0] is not None else np.nan
                chunk_map.append(map_score)
            except:
                chunk_map.append(np.nan)
        
        map_scores.extend(chunk_map)
    
    bw.close()
    
    # Add mappability scores to DataFrame
    genome_bins_with_map = genome_bins.copy()
    genome_bins_with_map['mappability'] = map_scores
    
    # Report on missing data
    missing_map = genome_bins_with_map['mappability'].isna().sum()
    total_bins = len(genome_bins_with_map)
    print(f"\nMissing mappability data summary:")
    print(f"Total bins: {total_bins}")
    print(f"Bins without mappability data: {missing_map} ({missing_map/total_bins*100:.2f}%)")
    print(f"\nMappability Score Summary:")
    print(genome_bins_with_map['mappability'].describe())
    
    return genome_bins_with_map

map_bigwig = "../data/hg38_ref_files/k100.Umap.MultiTrackMappability.bw"
map_data = load_mappability(map_bigwig, chunk_size=1000)
map_data.to_csv('../data/mappability.bed', sep='\t', index=False)

Processing bins 0 to 1000 of 56596 (0.0%)
Processing bins 1000 to 2000 of 56596 (1.8%)
Processing bins 2000 to 3000 of 56596 (3.5%)
Processing bins 3000 to 4000 of 56596 (5.3%)
Processing bins 4000 to 5000 of 56596 (7.1%)
Processing bins 5000 to 6000 of 56596 (8.8%)
Processing bins 6000 to 7000 of 56596 (10.6%)
Processing bins 7000 to 8000 of 56596 (12.4%)
Processing bins 8000 to 9000 of 56596 (14.1%)
Processing bins 9000 to 10000 of 56596 (15.9%)
Processing bins 10000 to 11000 of 56596 (17.7%)
Processing bins 11000 to 12000 of 56596 (19.4%)
Processing bins 12000 to 13000 of 56596 (21.2%)
Processing bins 13000 to 14000 of 56596 (23.0%)
Processing bins 14000 to 15000 of 56596 (24.7%)
Processing bins 15000 to 16000 of 56596 (26.5%)
Processing bins 16000 to 17000 of 56596 (28.3%)
Processing bins 17000 to 18000 of 56596 (30.0%)
Processing bins 18000 to 19000 of 56596 (31.8%)
Processing bins 19000 to 20000 of 56596 (33.6%)
Processing bins 20000 to 21000 of 56596 (35.3%)
Processing bins 2100

In [5]:
# Chromatin State (Heterochromatin)

def load_chromatin_states(chromatin_bed, chunk_size=1000):
    genome_bins = pd.read_csv('../data/genome_bins.bed', sep='\t')
    genome_bins_with_het = genome_bins.copy()

    heterochromatin_states = {'9_Het', '13_ReprPC', '14_ReprPCWk'}

    chrom_states = pd.read_csv(chromatin_bed, sep='\t', names=['chrom', 'start', 'end', 'name'])
    chrom_states['chrom'] = chrom_states['chrom'].str.replace('chr', '', regex=False)

    # filter to heterochromatin
    chrom_states = chrom_states[chrom_states["name"].isin(heterochromatin_states)].copy()

    # pre-split by chromosome
    chrom_states_by_chr = {
        c: dfc.sort_values("start") for c, dfc in chrom_states.groupby("chrom", sort=False)
    }

    genome_bins_with_het['frac_heterochromatin'] = 0.0

    total_bins = len(genome_bins_with_het)
    for i in range(0, total_bins, chunk_size):
        chunk = genome_bins_with_het.iloc[i:i+chunk_size].copy()
        print(f"Processing bins {i} to {i+len(chunk)} of {total_bins} ({(i/total_bins*100):.1f}%)")

        for j, bin_row in chunk.iterrows():
            bin_len = bin_row['end'] - bin_row['start']
            bin_chrom = str(bin_row['chrom'])

            # only look at regions on this chromosome
            chrom_df = chrom_states_by_chr.get(bin_chrom)
            if chrom_df is None or bin_len <= 0:
                genome_bins_with_het.at[j, 'frac_heterochromatin'] = 0.0
                continue

            # filter overlaps
            overlapping = chrom_df[
                (chrom_df['end'] > bin_row['start']) &
                (chrom_df['start'] < bin_row['end'])
            ]

            heterochromatin_length = 0
            for _, region in overlapping.iterrows():
                overlap_start = max(region['start'], bin_row['start'])
                overlap_end = min(region['end'], bin_row['end'])
                heterochromatin_length += max(0, overlap_end - overlap_start)

            genome_bins_with_het.at[j, 'frac_heterochromatin'] = heterochromatin_length / bin_len

    print(f"\nSummary statistics for heterochromatin fraction:")
    print(f"{genome_bins_with_het['frac_heterochromatin'].describe()}")
    print(f"Bins with >50% heterochromatin: {(genome_bins_with_het['frac_heterochromatin'] > 0.5).sum()}")
    print(f"Bins with >90% heterochromatin: {(genome_bins_with_het['frac_heterochromatin'] > 0.9).sum()}")

    return genome_bins_with_het

chrom_states_file = "../data/hg38_ref_files/E129_15_coreMarks_hg38lift_mnemonics.bed"
processed_data = load_chromatin_states(chrom_states_file, chunk_size=1000)
processed_data.to_csv('../data/chrom_states.bed', sep='\t', index=False)

Processing bins 0 to 1000 of 56596 (0.0%)
Processing bins 1000 to 2000 of 56596 (1.8%)
Processing bins 2000 to 3000 of 56596 (3.5%)
Processing bins 3000 to 4000 of 56596 (5.3%)
Processing bins 4000 to 5000 of 56596 (7.1%)
Processing bins 5000 to 6000 of 56596 (8.8%)
Processing bins 6000 to 7000 of 56596 (10.6%)
Processing bins 7000 to 8000 of 56596 (12.4%)
Processing bins 8000 to 9000 of 56596 (14.1%)
Processing bins 9000 to 10000 of 56596 (15.9%)
Processing bins 10000 to 11000 of 56596 (17.7%)
Processing bins 11000 to 12000 of 56596 (19.4%)
Processing bins 12000 to 13000 of 56596 (21.2%)
Processing bins 13000 to 14000 of 56596 (23.0%)
Processing bins 14000 to 15000 of 56596 (24.7%)
Processing bins 15000 to 16000 of 56596 (26.5%)
Processing bins 16000 to 17000 of 56596 (28.3%)
Processing bins 17000 to 18000 of 56596 (30.0%)
Processing bins 18000 to 19000 of 56596 (31.8%)
Processing bins 19000 to 20000 of 56596 (33.6%)
Processing bins 20000 to 21000 of 56596 (35.3%)
Processing bins 2100

In [6]:
def union_overlap_len(starts, ends):
    # returns total length of union of overlapping intervals (two intervals can cover some of the same bases)
    if len(starts) == 0:
        return 0
    order = np.argsort(starts)
    starts = np.asarray(starts)[order]
    ends = np.asarray(ends)[order]

    total = 0
    cur_s, cur_e = int(starts[0]), int(ends[0])
    for s, e in zip(starts[1:], ends[1:]):
        s = int(s); e = int(e)
        if s <= cur_e:
            cur_e = max(cur_e, e)
        else:
            total += max(0, cur_e - cur_s)
            cur_s, cur_e = s, e
    total += max(0, cur_e - cur_s)
    return total

REPEAT_CLASSES = {
    'frac_sine': 'SINE',
    'frac_line': 'LINE',
    'frac_ltr':  'LTR',
}

def load_repeat_masker(repeat_file, chunk_size=1000):

    genome_bins = pd.read_csv('../data/genome_bins.bed', sep='\t')
    genome_bins['chrom'] = genome_bins['chrom'].astype(str)
    genome_bins_with_repeats = genome_bins.copy()
    for col in REPEAT_CLASSES:
        genome_bins_with_repeats[col] = 0.0

    repeats_df = pd.read_csv(
        repeat_file, sep='\t', comment='#',
        names=['bin','swScore','milliDiv','milliDel','milliIns','genoName','genoStart','genoEnd','genoLeft','strand',
               'repName','repClass','repFamily','repStart','repEnd','repLeft','id'],
        low_memory=False
    )
    repeats_df['genoName'] = repeats_df['genoName'].astype(str).str.replace('chr', '', regex=False)

    # Filter for included repeat classes
    target_classes = set(REPEAT_CLASSES.values())
    repeats_df = repeats_df[repeats_df['repClass'].isin(target_classes)].copy()
    repeats_df = repeats_df[repeats_df['genoName'].isin(genome_bins['chrom'].unique())]

    # Pre-split by chromosome and repClass for fast lookup
    repeats_by_chr_class = {}
    for (chrom, cls), dfc in repeats_df.groupby(['genoName', 'repClass'], sort=False):
        repeats_by_chr_class[(chrom, cls)] = dfc.sort_values(['genoStart', 'genoEnd'])

    total_bins = len(genome_bins_with_repeats)
    for i in range(0, total_bins, chunk_size):
        chunk = genome_bins_with_repeats.iloc[i:i+chunk_size].copy()
        print(f"Processing bins {i} to {i+len(chunk)} of {total_bins} ({(i/total_bins*100):.1f}%)")

        for j, bin_row in chunk.iterrows():
            bin_len = bin_row['end'] - bin_row['start']
            if bin_len <= 0:
                continue

            chrom = str(bin_row['chrom'])

            for col, cls in REPEAT_CLASSES.items():
                chrom_cls_repeats = repeats_by_chr_class.get((chrom, cls))
                if chrom_cls_repeats is None:
                    continue

                overlapping = chrom_cls_repeats[
                    (chrom_cls_repeats['genoEnd']   > bin_row['start']) &
                    (chrom_cls_repeats['genoStart'] < bin_row['end'])
                ]

                clip_starts = np.maximum(overlapping['genoStart'].to_numpy(), bin_row['start'])
                clip_ends   = np.minimum(overlapping['genoEnd'].to_numpy(),   bin_row['end'])
                repeat_len  = union_overlap_len(clip_starts, clip_ends)

                genome_bins_with_repeats.at[j, col] = repeat_len / bin_len

    print("\nSummary of repeat fractions:")
    print(genome_bins_with_repeats[list(REPEAT_CLASSES.keys())].describe())
    return genome_bins_with_repeats

repeat_file = '../data/hg38_ref_files/repeats.tsv'
repeat_features = load_repeat_masker(repeat_file, chunk_size=1000)
repeat_features.to_csv('../data/repeat_features.csv', index=False)

Processing bins 0 to 1000 of 56596 (0.0%)
Processing bins 1000 to 2000 of 56596 (1.8%)
Processing bins 2000 to 3000 of 56596 (3.5%)
Processing bins 3000 to 4000 of 56596 (5.3%)
Processing bins 4000 to 5000 of 56596 (7.1%)
Processing bins 5000 to 6000 of 56596 (8.8%)
Processing bins 6000 to 7000 of 56596 (10.6%)
Processing bins 7000 to 8000 of 56596 (12.4%)
Processing bins 8000 to 9000 of 56596 (14.1%)
Processing bins 9000 to 10000 of 56596 (15.9%)
Processing bins 10000 to 11000 of 56596 (17.7%)
Processing bins 11000 to 12000 of 56596 (19.4%)
Processing bins 12000 to 13000 of 56596 (21.2%)
Processing bins 13000 to 14000 of 56596 (23.0%)
Processing bins 14000 to 15000 of 56596 (24.7%)
Processing bins 15000 to 16000 of 56596 (26.5%)
Processing bins 16000 to 17000 of 56596 (28.3%)
Processing bins 17000 to 18000 of 56596 (30.0%)
Processing bins 18000 to 19000 of 56596 (31.8%)
Processing bins 19000 to 20000 of 56596 (33.6%)
Processing bins 20000 to 21000 of 56596 (35.3%)
Processing bins 2100

In [7]:
# Fragile sites

def process_fragile_sites(fragile_sites_dir):

    genome_bins = pd.read_csv('../data/genome_bins.bed', sep='\t')
    genome_bins['chrom'] = genome_bins['chrom'].astype(str)
    genome_bins_with_fragile = genome_bins.copy()
    genome_bins_with_fragile['frac_fragile_site'] = 0.0
    
    # Process each chromosome's fragile site file
    for chrom in genome_bins_with_fragile['chrom'].unique():
        print(f"Processing chromosome {chrom}")
        
        # Get bins for this chromosome
        chrom_bins = genome_bins_with_fragile[genome_bins_with_fragile['chrom'] == chrom].copy()
        
        # Read fragile site file for this chromosome
        fragile_file = os.path.join(fragile_sites_dir, f"chr{chrom}_fragile_site.bed")
        if not os.path.exists(fragile_file):
            print(f"No fragile site file found for chr{chrom}")
            continue
            
        # Read fragile sites for this chromosome
        fragile_df = pd.read_csv(fragile_file, sep='\t', names=['chrom', 'start', 'end', 'name', 'score', 'strand'])
        print(f"Number of fragile sites in chr{chrom}: {len(fragile_df)}")
        
        # for each bin on this chromosome, compute union overlap of all fragile sites
        for bin_idx, bin_row in chrom_bins.iterrows():
            bin_size = bin_row["end"] - bin_row["start"]

            overlapping_sites = fragile_df[
                (fragile_df["end"] > bin_row["start"]) &
                (fragile_df["start"] < bin_row["end"])
            ]
            if overlapping_sites.empty:
                continue

            clip_starts = np.maximum(overlapping_sites["start"].to_numpy(), bin_row["start"])
            clip_ends   = np.minimum(overlapping_sites["end"].to_numpy(),   bin_row["end"])

            fragile_len = union_overlap_len(clip_starts, clip_ends)
            genome_bins_with_fragile.at[bin_idx, "frac_fragile_site"] = fragile_len / bin_size

    print("\nSummary of fragile site fractions:")
    print(genome_bins_with_fragile['frac_fragile_site'].describe())
    print(f"Number of bins with non-zero fragile site overlap: {(genome_bins_with_fragile['frac_fragile_site'] > 0).sum()}")
    
    genome_bins_with_fragile.to_csv('../data/fragile_sites.csv', index=False)
    
    return genome_bins_with_fragile

fragile_sites = process_fragile_sites(
    fragile_sites_dir='../data/hg38_ref_files/fragile_site_bed'
)

Processing chromosome 1
Number of fragile sites in chr1: 13
Processing chromosome 2
Number of fragile sites in chr2: 13
Processing chromosome 3
Number of fragile sites in chr3: 4
Processing chromosome 4
Number of fragile sites in chr4: 5
Processing chromosome 5
Number of fragile sites in chr5: 8
Processing chromosome 6
Number of fragile sites in chr6: 8
Processing chromosome 7
Number of fragile sites in chr7: 11
Processing chromosome 8
Number of fragile sites in chr8: 5
Processing chromosome 9
Number of fragile sites in chr9: 6
Processing chromosome 10
Number of fragile sites in chr10: 7
Processing chromosome 11
Number of fragile sites in chr11: 9
Processing chromosome 12
Number of fragile sites in chr12: 5
Processing chromosome 13
Number of fragile sites in chr13: 5
Processing chromosome 14
Number of fragile sites in chr14: 2
Processing chromosome 15
Number of fragile sites in chr15: 1
Processing chromosome 16
Number of fragile sites in chr16: 5
Processing chromosome 17
Number of frag

In [8]:
def process_gene_density(gene_file):
    
    genome_bins = pd.read_csv('../data/genome_bins.bed', sep='\t')
    genome_bins['chrom'] = genome_bins['chrom'].astype(str)
    genome_bins_with_genes = genome_bins.copy()
    genome_bins_with_genes['gene_count'] = 0
    genome_bins_with_genes['gene_density'] = 0.0

    genes_df = pd.read_csv(
        gene_file, sep='\t',
        names=['chrom', 'start', 'end', 'clusterId', 'transcript', 'protein'],
        comment='#',
        compression='gzip',
        low_memory=False
    )
    # Filter to primary chromosomes
    primary_chroms = [f'chr{i}' for i in list(range(1,23))] + ['chrX', 'chrY']
    genes_df = genes_df[genes_df['chrom'].isin(primary_chroms)].copy()

    # One entry per gene cluster
    genes_df = genes_df.drop_duplicates(subset=['clusterId'])
    print(f"Unique gene clusters: {len(genes_df)}")

    genes_df['chrom'] = genes_df['chrom'].astype(str).str.replace('chr', '', regex=False).str.strip()
    genome_bins['chrom'] = genome_bins['chrom'].astype(str).str.strip()

    genes_by_chr = {
        c: dfc.sort_values('start')
        for c, dfc in genes_df.groupby('chrom', sort=False)
    }

    for chrom in genome_bins_with_genes['chrom'].unique():
        print(f"Processing chromosome {chrom}")
        chrom_bins  = genome_bins_with_genes[genome_bins_with_genes['chrom'] == chrom]
        chrom_genes = genes_by_chr.get(chrom)
        if chrom_genes is None:
            print(f"No genes found for chr{chrom}")
            continue

        for bin_idx, bin_row in chrom_bins.iterrows():
            bin_size_mb = (bin_row['end'] - bin_row['start']) / 1e6
            overlapping = chrom_genes[
                (chrom_genes['end']   > bin_row['start']) &
                (chrom_genes['start'] < bin_row['end'])
            ]
            n_genes = len(overlapping)
            genome_bins_with_genes.at[bin_idx, 'gene_count']   = n_genes
            genome_bins_with_genes.at[bin_idx, 'gene_density'] = n_genes / bin_size_mb

    print("\nSummary of gene density:")
    print(genome_bins_with_genes[['gene_count', 'gene_density']].describe())
    print(f"Bins with at least one gene: {(genome_bins_with_genes['gene_count'] > 0).sum()}")

    genome_bins_with_genes.to_csv('../data/gene_density.csv', index=False)
    return genome_bins_with_genes

gene_density = process_gene_density(
    gene_file='../data/hg38_ref_files/genes.bed.gz'
)

Unique gene clusters: 78654
Processing chromosome 1
Processing chromosome 2
Processing chromosome 3
Processing chromosome 4
Processing chromosome 5
Processing chromosome 6
Processing chromosome 7
Processing chromosome 8
Processing chromosome 9
Processing chromosome 10
Processing chromosome 11
Processing chromosome 12
Processing chromosome 13
Processing chromosome 14
Processing chromosome 15
Processing chromosome 16
Processing chromosome 17
Processing chromosome 18
Processing chromosome 19
Processing chromosome 20
Processing chromosome 21
Processing chromosome 22
Processing chromosome X
Processing chromosome Y

Summary of gene density:
         gene_count  gene_density
count  56596.000000  56596.000000
mean       2.113506     42.270125
std        1.775872     35.517438
min        0.000000      0.000000
25%        1.000000     20.000000
50%        2.000000     40.000000
75%        3.000000     60.000000
max       42.000000    840.000000
Bins with at least one gene: 50305
